# Study 903 — Sector-Neutral Low-Vol — the teardown

The raw-vs-neutral spread splits, the per-leg Sharpe race, the defensive-tilt diagnostic, the Newey-West spread *t*, the 1,000-permutation placebo, the two-era cut, the costed timer, and the synthetic control (null + planted + the sector-confound proof).

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_rows': 4147, 'n_days': 4083, 'fingerprint': '357fd262912f', 'raw_spread_bps': -4.74, 'raw_t_nw': -2.61, 'raw_t_1s': -2.47, 'raw_lo_bps': 5.21, 'raw_hi_bps': 9.95, 'raw_welch': -1.66, 'raw_sharpe': -0.61, 'raw_lo_sh': 0.98, 'raw_hi_sh': 0.98, 'raw_lo_vol': 0.13, 'raw_hi_vol': 0.26, 'raw_long_def': 44.7, 'raw_short_def': 5.2, 'raw_ls_def': 39.5, 'univ_def': 20.0, 'neu_spread_bps': -3.53, 'neu_t_nw': -2.67, 'neu_t_1s': -2.52, 'neu_lo_bps': 6.54, 'neu_hi_bps': 10.08, 'neu_welch': -1.26, 'neu_sharpe': -0.63, 'neu_lo_sh': 1.05, 'neu_hi_sh': 1.07, 'neu_lo_vol': 0.16, 'neu_hi_vol': 0.24, 'neu_long_def': 13.9, 'neu_short_def': 15.7, 'neu_ls_def': -1.8, 'delta_spread': 1.2, 'placebo_obs': -3.53, 'placebo_mean': -0.005, 'placebo_sd': 0.993, 'placebo_sigma': 3.55, 'placebo_p': 1.0, 'placebo_draws': 1000, 'era_early_bps': -3.08, 'era_early_t': -1.93, 'era_early_n': 1949, 'era_late_bps': -3.95, 'era_late_t': -1.9, 'era_late_n': 2134, 'timer_1_gross': -3.53, 'timer_1_cost': 2.14, 'timer_1_net': -5.67, 'timer_1_t': -4.04, 'timer_1_sh': -1.0, 'timer_1_ann': -14.3, 'timer_5_gross': -3.53, 'timer_5_cost': 10.14, 'timer_5_net': -13.67, 'timer_5_t': -9.74, 'timer_5_sh': -2.42, 'timer_5_ann': -34.4, 'null_mean_t': 0.2, 'null_sd_t': 0.64, 'null_fire': 0, 'planted_t': 4.23, 'planted_welch': 4.35, 'planted_edge': 0.1, 'confound_raw_t': 3.46, 'confound_raw_fire': 18, 'confound_neu_fire': 0}

## The headline — raw vs sector-neutral low-vol spread

Daily equal-weight bottom-30% minus top-30% trailing-63d-vol spread (long low-vol, short high-vol), on the same panel.

In [2]:
print(f"RAW    spread : {R['raw_spread_bps']:+.2f} bps/day  NW(10) t = {R['raw_t_nw']:+.2f}  "
      f"one-sample t = {R['raw_t_1s']:+.2f}  (Welch {R['raw_welch']:+.2f})")
print(f"NEUTRAL spread: {R['neu_spread_bps']:+.2f} bps/day  NW(10) t = {R['neu_t_nw']:+.2f}  "
      f"one-sample t = {R['neu_t_1s']:+.2f}  (Welch {R['neu_welch']:+.2f})")
print(f"stripping the sector bet moved the spread {R['delta_spread']:+.2f} bps -> still wrong-signed")

RAW    spread : -4.74 bps/day  NW(10) t = -2.61  one-sample t = -2.47  (Welch -1.66)
NEUTRAL spread: -3.53 bps/day  NW(10) t = -2.67  one-sample t = -2.52  (Welch -1.26)
stripping the sector bet moved the spread +1.20 bps -> still wrong-signed


## The per-leg Sharpe race — the *risk-adjusted* low-vol claim

The anomaly is really about return **per unit of risk**. Each leg's own Sharpe (≈ excess-of-cash on a daily book), with its annualised vol.

In [3]:
print(f"RAW    : low-vol Sharpe {R['raw_lo_sh']:.2f} (vol {R['raw_lo_vol']:.2f}) vs "
      f"high-vol {R['raw_hi_sh']:.2f} (vol {R['raw_hi_vol']:.2f})  -> a tie")
print(f"NEUTRAL: low-vol Sharpe {R['neu_lo_sh']:.2f} (vol {R['neu_lo_vol']:.2f}) vs "
      f"high-vol {R['neu_hi_sh']:.2f} (vol {R['neu_hi_vol']:.2f})  -> high-vol edges it")

RAW    : low-vol Sharpe 0.98 (vol 0.13) vs high-vol 0.98 (vol 0.26)  -> a tie
NEUTRAL: low-vol Sharpe 1.05 (vol 0.16) vs high-vol 1.07 (vol 0.24)  -> high-vol edges it


## The defensive-sector tilt — what the raw sort actually buys

In [4]:
print(f"RAW    : long book {R['raw_long_def']:.1f}% defensive vs short {R['raw_short_def']:.1f}% "
      f"(universe {R['univ_def']:.1f}%)  -> long-short tilt {R['raw_ls_def']:+.1f}%")
print(f"NEUTRAL: long book {R['neu_long_def']:.1f}% defensive vs short {R['neu_short_def']:.1f}% "
      f"-> long-short tilt {R['neu_ls_def']:+.1f}% (neutralised)")

RAW    : long book 44.7% defensive vs short 5.2% (universe 20.0%)  -> long-short tilt +39.5%
NEUTRAL: long book 13.9% defensive vs short 15.7% -> long-short tilt -1.8% (neutralised)


## Placebo — column-permute the forward returns (1,000 permutations, sector-neutral)

In [5]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> ~{R['placebo_sigma']:.2f} sigma into the LEFT tail "
      f"(right-tail p = {R['placebo_p']:.4f})")

observed -3.53 bps vs placebo mean -0.005 (sd 0.993) -> ~3.55 sigma into the LEFT tail (right-tail p = 1.0000)


## Robustness — two eras (split 2018-01-01, sector-neutral)

In [6]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")

2010-2017 (n=1949): -3.08 bps  NW t = -1.93
2018-2026 (n=2134): -3.95 bps  NW t = -1.90


## The timer — can you get paid for it (sector-neutral book)?

2 sides × one-way cost × NAV per day on the long-short book; short (high-vol) pays 50 bps/yr borrow.

In [7]:
for tag,g,c,n,t,sh in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t'],R['timer_1_sh']),
                       ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'],R['timer_5_sh'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f}, Sharpe {sh:.2f})")

 1 bp one-way: gross -3.53 -> net -5.67 bps/day (cost 2.14/day, t=-4.04, Sharpe -1.00)
5 bps one-way: gross -3.53 -> net -13.67 bps/day (cost 10.14/day, t=-9.74, Sharpe -2.42)


## Synthetic positive control — the machinery is sound

Live: the sector-neutral detector must NOT fire on the null, must recover a planted within-sector low-vol effect, and — crucially — a pure sector premium must fool the RAW sort but leave the NEUTRAL sort silent.

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np, pandas as pd
from sn_lowvol import data, strategy as st
secmap = pd.Series(data.synthetic_sectors(40, 8))
null_t = np.array([st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=903+s, n_assets=40, n_days=1200, n_sectors=8, sector_prem_ann=0.0), secmap, neutral=True)['t_nw'] for s in range(8)])
print(f"null (edge=0), 8 seeds, neutral: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(edge=0.1, seed=903, n_assets=40, n_days=1500, n_sectors=8), secmap, neutral=True)
print(f"planted within-sector low-vol (edge=0.1), neutral: NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")
conf = data.synthetic_panel(edge=0.0, seed=903, n_assets=40, n_days=1500, n_sectors=8, sector_prem_ann=0.08)
rc = st.close_returns(conf)
raw_t = st.vol_stats(st.vol_spreads(rc, secmap, 63, 0.3, neutral=False))['t_nw']
neu_t = st.vol_stats(st.vol_spreads(rc, secmap, 63, 0.3, neutral=True))['t_nw']
print(f"confound (sector premium only): RAW fires (t={raw_t:+.2f}) but NEUTRAL silent (t={neu_t:+.2f})")

null (edge=0), 8 seeds, neutral: NW t mean +0.46 (sd 0.50), |t|>=2 in 0/8


planted within-sector low-vol (edge=0.1), neutral: NW t = +4.23, Welch t = +4.35


confound (sector premium only): RAW fires (t=+3.13) but NEUTRAL silent (t=-0.53)


## Verdict

- **Signal — None.** The low-vol edge does **not** survive on 50 liquid US mega-caps. The sector-neutral long-low-vol / short-high-vol spread is **-3.53 bps/day** (NW *t* = **-2.67**) — significant but *opposite in sign* to the claim (the wild tech names out-earned), holding in both eras (*t* = -1.93 / -1.90) and ≈3.5σ into the left tail of a 1,000-permutation placebo. On the anomaly's own **Sharpe** axis the low-vol leg has no advantage once sector-neutral (1.05 vs 1.07). The naive sort's character was largely a **defensive-sector tilt** (long book 45% defensive; neutralising moved the spread +1.20 bps), and the synthetic control shows a pure sector premium fools the raw sort (*t* = +3.46) while the neutral sort stays silent (0/8) — a clean null, not machinery.
- **Tradability — Mirage.** The sector-neutral book loses money gross and net (**-5.67 bps/day** at 1 bp one-way, -13.67 at 5 bps); even the data-mined sign-flip is eaten by the 2.14 bps/day round-trip friction. *Survivorship: current-membership mega-caps — an upper bound.*